
# Narrative Signal による RRG 状態遷移の予測・説明分析

このノートブックは、テーマ別の **Narrative Signal** が、RRG 上のテーマ・ライフサイクル状態遷移を予測・説明できるかを検証します。

中核仮説は次です。

\[
NarrativeSignal_{g,t}
\Rightarrow
Pr(S_{g,t+h}=s' \mid S_{g,t}=s)
\]

特に重要なのは、RRG の **Improving → Leading** 遷移です。

\[
Pr(S_{g,t+h}=Leading \mid S_{g,t}=Improving)
\]

この確率を、Narrative Signal、APWC、Residual Momentum、現在の RRG 座標で説明・予測します。

想定する入力ファイルは以下です。

```text
DATA_DIR/
  daily_narrative_intensity.csv
  theme_residual_returns.csv
  theme_apwc.csv
  theme_rrg.csv  # 任意。なければ残差リターンからRRGを計算
```

入力が存在しない場合は synthetic demo data を自動生成して、ノートブック全体が動くようにしています。



## 1. 設定

ここでは、入力データの場所、narrative shock の窓、narrative beta 推定窓、状態遷移の予測ホライズンなどを指定します。

実データで使う場合は、`DATA_DIR` を変更し、`USE_SYNTHETIC_IF_MISSING=False` にしてください。


In [ ]:

from dataclasses import dataclass
from pathlib import Path

@dataclass
class Config:
    DATA_DIR: str = "/mnt/data/rrg_transition_input"
    USE_SYNTHETIC_IF_MISSING: bool = True
    SAVE_OUTPUTS: bool = False
    OUTPUT_DIR: str = "/mnt/data/rrg_transition_outputs"

    # Narrative intensity / shock
    INTENSITY_COLUMN_PREFERENCE: tuple = ("I_negative", "I_total", "intensity", "I")
    SHOCK_WINDOWS: tuple = (20, 60, 120)
    MAIN_SHOCK_WINDOW: int = 60
    ZSCORE_WINDOW: int = 126
    ZSCORE_MIN_PERIODS: int = 63

    # Narrative beta estimation
    BETA_WINDOW: int = 126
    BETA_MIN_OBS: int = 63
    RIDGE_ALPHA: float = 10.0
    BETA_REFIT_STEP: int = 20  # rolling betaを何営業日ごとに再推定するか。1なら毎日。
    MAX_NARRATIVES_FOR_BETA: int = 20  # 多すぎる場合はボラティリティ上位などに絞る

    # RRG
    RRG_RS_WINDOW: int = 63
    RRG_MOM_LAG: int = 20
    RRG_HYSTERESIS_BAND: float = 0.0  # 最初は0。ノイズが強ければ0.1などにする。

    # Returns / APWC / Momentum
    RMOM_WINDOWS: tuple = (20, 60, 120)
    VOL_WINDOW: int = 60
    APWC_Z_WINDOW: int = 126

    # Transition prediction horizons
    HORIZONS: tuple = (5, 20, 60)
    MAIN_HORIZON: int = 20

    # OOS walk-forward
    TRAIN_MIN_DAYS: int = 252
    TEST_STEP_DAYS: int = 126
    OOS_MAX_FOLDS: int = 4  # synthetic/demo実行を速くするための上限。実分析ではNone推奨。

    INCLUDE_THEME_FE: bool = False
    INCLUDE_MONTH_FE: bool = False

    RANDOM_SEED: int = 42

CONFIG = Config()
Path(CONFIG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
CONFIG



## 2. ライブラリとユーティリティ関数

このセルでは、分析に使うライブラリと共通関数を定義します。

主な関数は以下です。

- rolling z-score
- 横断面 z-score
- RRG 状態判定
- 回帰・分類モデルの安全な実行
- forward return / state transition 作成


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    confusion_matrix, classification_report
)
from sklearn.cluster import KMeans

import statsmodels.api as sm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


def rolling_zscore(s: pd.Series, window: int, min_periods: int = None) -> pd.Series:
    if min_periods is None:
        min_periods = max(10, window // 2)
    m = s.rolling(window, min_periods=min_periods).mean()
    sd = s.rolling(window, min_periods=min_periods).std()
    return (s - m) / sd.replace(0, np.nan)


def xsec_zscore(df: pd.DataFrame, value_col: str, date_col: str = "date") -> pd.Series:
    def _z(x):
        sd = x.std(ddof=0)
        if sd == 0 or np.isnan(sd):
            return pd.Series(np.zeros(len(x)), index=x.index)
        return (x - x.mean()) / sd
    return df.groupby(date_col)[value_col].transform(_z)


def safe_qcut(s: pd.Series, q=2, labels=None):
    try:
        return pd.qcut(s, q=q, labels=labels, duplicates="drop")
    except Exception:
        return pd.Series([np.nan] * len(s), index=s.index)


def assign_rrg_state(rsratio, rsmom, band=0.0):
    # band=0なら標準的な4象限。band>0なら小さな境界ノイズを減らせる。
    if pd.isna(rsratio) or pd.isna(rsmom):
        return np.nan
    if rsratio > band and rsmom > band:
        return "Leading"
    if rsratio > band and rsmom <= -band:
        return "Weakening"
    if rsratio <= -band and rsmom <= -band:
        return "Lagging"
    if rsratio <= -band and rsmom > band:
        return "Improving"
    # band 内は最も近い象限に割り当てる
    if rsratio >= 0 and rsmom >= 0:
        return "Leading"
    if rsratio >= 0 and rsmom < 0:
        return "Weakening"
    if rsratio < 0 and rsmom < 0:
        return "Lagging"
    return "Improving"


def add_constant_and_fit_glm_binomial(df, y_col, x_cols, add_theme_fe=False, add_month_fe=False):
    use_cols = [y_col] + x_cols + ["theme_id", "date"]
    d = df[use_cols].dropna().copy()
    if d.empty or d[y_col].nunique() < 2:
        return None, d
    X = d[x_cols].copy()
    # 標準化。ダミー以外の数値列を対象。
    for c in x_cols:
        sd = X[c].std()
        if sd and not np.isnan(sd):
            X[c] = (X[c] - X[c].mean()) / sd
        else:
            X[c] = 0.0
    if add_theme_fe:
        X = pd.concat([X, pd.get_dummies(d["theme_id"], prefix="theme", drop_first=True, dtype=float)], axis=1)
    if add_month_fe:
        month = pd.to_datetime(d["date"]).dt.to_period("M").astype(str)
        X = pd.concat([X, pd.get_dummies(month, prefix="month", drop_first=True, dtype=float)], axis=1)
    X = sm.add_constant(X, has_constant="add")
    y = d[y_col].astype(float)
    try:
        model = sm.GLM(y, X, family=sm.families.Binomial()).fit(cov_type="HC1")
        return model, d
    except Exception as e:
        print(f"GLM failed for {y_col}: {e}")
        return None, d


def fit_ols_hc1(df, y_col, x_cols, add_theme_fe=False, add_month_fe=False):
    use_cols = [y_col] + x_cols + ["theme_id", "date"]
    d = df[use_cols].dropna().copy()
    if d.empty:
        return None, d
    X = d[x_cols].copy()
    for c in x_cols:
        sd = X[c].std()
        if sd and not np.isnan(sd):
            X[c] = (X[c] - X[c].mean()) / sd
        else:
            X[c] = 0.0
    if add_theme_fe:
        X = pd.concat([X, pd.get_dummies(d["theme_id"], prefix="theme", drop_first=True, dtype=float)], axis=1)
    if add_month_fe:
        month = pd.to_datetime(d["date"]).dt.to_period("M").astype(str)
        X = pd.concat([X, pd.get_dummies(month, prefix="month", drop_first=True, dtype=float)], axis=1)
    X = sm.add_constant(X, has_constant="add")
    y = d[y_col].astype(float)
    try:
        model = sm.OLS(y, X).fit(cov_type="HC1")
        return model, d
    except Exception as e:
        print(f"OLS failed for {y_col}: {e}")
        return None, d


def print_key_coefs(model, keys=None, top=20):
    if model is None:
        print("Model is None.")
        return
    res = pd.DataFrame({
        "coef": model.params,
        "t_or_z": model.tvalues,
        "pvalue": model.pvalues,
    })
    if keys is not None:
        display(res.loc[[k for k in keys if k in res.index]])
    else:
        display(res.head(top))


def ann_stats(ret: pd.Series, periods=252):
    r = ret.dropna()
    if len(r) == 0:
        return pd.Series({"ann_return": np.nan, "ann_vol": np.nan, "sharpe": np.nan, "max_drawdown": np.nan, "hit_rate": np.nan, "n": 0})
    ann_return = r.mean() * periods
    ann_vol = r.std() * np.sqrt(periods)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
    cum = (1 + r).cumprod()
    dd = cum / cum.cummax() - 1
    return pd.Series({
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": dd.min(),
        "hit_rate": (r > 0).mean(),
        "n": len(r),
    })



## 3. データ読み込みまたは synthetic demo data 生成

実データがある場合はCSVを読み込みます。ない場合は、narrative intensity、テーマ残差リターン、APWC が互いにある程度関係するような synthetic data を生成します。

synthetic data はロジック確認用です。経済的な結論には使わないでください。


In [ ]:

DATA_DIR = Path(CONFIG.DATA_DIR)


def file_exists(name):
    return (DATA_DIR / name).exists()


def generate_synthetic_data(config: Config):
    rng = np.random.default_rng(config.RANDOM_SEED)
    dates = pd.bdate_range("2020-01-02", "2024-12-31")
    n_dates = len(dates)
    themes = [f"Theme_{i:02d}" for i in range(1, 9)]
    narratives = [f"Narrative_{j:02d}" for j in range(1, 13)]

    # Narrative intensity: AR processes with occasional pulses.
    intensity_records = []
    for j, n in enumerate(narratives):
        x = np.zeros(n_dates)
        phi = 0.96 + 0.02 * rng.random()
        noise = rng.normal(0, 0.004, n_dates)
        base = 0.01 + 0.01 * rng.random()
        x[0] = base
        for t in range(1, n_dates):
            x[t] = base * (1 - phi) + phi * x[t-1] + noise[t]
        # pulses
        for _ in range(4):
            center = rng.integers(100, n_dates-100)
            width = rng.integers(20, 90)
            amp = rng.uniform(0.02, 0.08)
            pulse = amp * np.exp(-0.5 * ((np.arange(n_dates) - center) / width) ** 2)
            x += pulse
        x = np.clip(x, 0, None)
        for d, val in zip(dates, x):
            intensity_records.append({"date": d, "narrative_id": n, "intensity": val})
    intensity = pd.DataFrame(intensity_records)

    # Precompute shocks for data generation.
    piv = intensity.pivot(index="date", columns="narrative_id", values="intensity").sort_index()
    shock = piv.rolling(60).mean().shift(1) - piv.rolling(60).mean().shift(61)
    zshock = shock.apply(lambda s: rolling_zscore(s, 252, 126))
    zshock = zshock.fillna(0.0)

    # Theme betas sparse.
    beta_true = pd.DataFrame(0.0, index=themes, columns=narratives)
    for th in themes:
        idx = rng.choice(len(narratives), size=4, replace=False)
        beta_true.loc[th, [narratives[i] for i in idx]] = rng.normal(0.0008, 0.0012, size=4)

    # residual returns generated from narrative shocks + own AR momentum + noise.
    ret_records = []
    apwc_records = []
    rrg_records = []
    ret_mat = pd.DataFrame(index=dates, columns=themes, dtype=float)
    for th in themes:
        signal = zshock.dot(beta_true.loc[th]).values
        eps = rng.normal(0, 0.008, n_dates)
        r = np.zeros(n_dates)
        for t in range(1, n_dates):
            r[t] = 0.08 * r[t-1] + signal[t] + eps[t]
        ret_mat[th] = r
        for d, val in zip(dates, r):
            ret_records.append({"date": d, "theme_id": th, "residual_return": val})

    # APWC higher when absolute narrative signal and residual momentum are high.
    for th in themes:
        r = ret_mat[th]
        rmom = r.rolling(60).sum()
        signal_abs = np.abs(zshock.dot(beta_true.loc[th])).rolling(20).mean()
        apwc = 0.08 + 0.05 * rolling_zscore(signal_abs.fillna(0), 252, 126).fillna(0) + 0.03 * rolling_zscore(rmom.fillna(0), 252, 126).fillna(0) + rng.normal(0, 0.04, n_dates)
        apwc = np.clip(apwc, -0.2, 0.8)
        for d, val in zip(dates, apwc):
            apwc_records.append({"date": d, "theme_id": th, "APWC": val})

    residual_returns = pd.DataFrame(ret_records)
    apwc = pd.DataFrame(apwc_records)
    return intensity, residual_returns, apwc, None

if file_exists("daily_narrative_intensity.csv") and file_exists("theme_residual_returns.csv") and file_exists("theme_apwc.csv"):
    daily_intensity_raw = pd.read_csv(DATA_DIR / "daily_narrative_intensity.csv")
    theme_returns_raw = pd.read_csv(DATA_DIR / "theme_residual_returns.csv")
    theme_apwc_raw = pd.read_csv(DATA_DIR / "theme_apwc.csv")
    theme_rrg_raw = pd.read_csv(DATA_DIR / "theme_rrg.csv") if file_exists("theme_rrg.csv") else None
    print("Loaded real input files.")
else:
    if not CONFIG.USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError("Input files not found and USE_SYNTHETIC_IF_MISSING=False.")
    daily_intensity_raw, theme_returns_raw, theme_apwc_raw, theme_rrg_raw = generate_synthetic_data(CONFIG)
    print("Generated synthetic demo data.")

print("daily_intensity_raw", daily_intensity_raw.shape)
print("theme_returns_raw", theme_returns_raw.shape)
print("theme_apwc_raw", theme_apwc_raw.shape)
if theme_rrg_raw is not None:
    print("theme_rrg_raw", theme_rrg_raw.shape)

display(daily_intensity_raw.head())
display(theme_returns_raw.head())
display(theme_apwc_raw.head())



## 4. Narrative intensity から shock / zshock を作成

ここでは daily narrative intensity から、Narrative Momentum 的な attention shock を作ります。

\[
Shock^J_{n,t}
= \overline{I}_{n,t-J:t-1} - \overline{I}_{n,t-2J:t-J-1}
\]

その後、narrativeごとに rolling z-score 化します。

この `ZShock` が narrative beta 方式の入力になります。


In [ ]:

# Normalize narrative intensity input
ni = daily_intensity_raw.copy()
ni.columns = [c.strip() for c in ni.columns]
ni["date"] = pd.to_datetime(ni["date"])

if "narrative_id" not in ni.columns:
    # try common alternatives
    for alt in ["narrative", "topic", "topic_id", "theme"]:
        if alt in ni.columns:
            ni = ni.rename(columns={alt: "narrative_id"})
            break

intensity_col = None
for c in CONFIG.INTENSITY_COLUMN_PREFERENCE:
    if c in ni.columns:
        intensity_col = c
        break
if intensity_col is None:
    numeric_cols = [c for c in ni.columns if c not in ["date", "narrative_id"] and np.issubdtype(ni[c].dtype, np.number)]
    if not numeric_cols:
        raise ValueError("No intensity column found.")
    intensity_col = numeric_cols[0]

ni = ni[["date", "narrative_id", intensity_col]].rename(columns={intensity_col: "intensity"})
ni = ni.dropna(subset=["date", "narrative_id", "intensity"])
ni = ni.sort_values(["narrative_id", "date"])

# Pivot and optionally keep top narratives by average absolute daily changes if too many.
piv = ni.pivot_table(index="date", columns="narrative_id", values="intensity", aggfunc="sum").sort_index()
piv = piv.fillna(0.0)

if piv.shape[1] > CONFIG.MAX_NARRATIVES_FOR_BETA:
    activity = piv.diff().abs().mean().sort_values(ascending=False)
    keep = activity.head(CONFIG.MAX_NARRATIVES_FOR_BETA).index.tolist()
    piv = piv[keep]
    print(f"Kept top {len(keep)} narratives by average absolute daily intensity change.")

shock_dfs = []
for w in CONFIG.SHOCK_WINDOWS:
    recent = piv.shift(1).rolling(w, min_periods=max(5, w//2)).mean()
    prev = piv.shift(1 + w).rolling(w, min_periods=max(5, w//2)).mean()
    shock = recent - prev
    zshock = shock.apply(lambda s: rolling_zscore(s, CONFIG.ZSCORE_WINDOW, CONFIG.ZSCORE_MIN_PERIODS))
    zshock_long = zshock.stack().rename(f"ZShock_{w}D").reset_index().rename(columns={"level_1": "narrative_id"})
    shock_long = shock.stack().rename(f"Shock_{w}D").reset_index().rename(columns={"level_1": "narrative_id"})
    tmp = shock_long.merge(zshock_long, on=["date", "narrative_id"], how="outer")
    shock_dfs.append(tmp)

narr_features = shock_dfs[0]
for tmp in shock_dfs[1:]:
    narr_features = narr_features.merge(tmp, on=["date", "narrative_id"], how="outer")

# Add level z-score for possible diagnostics.
level_z = piv.apply(lambda s: rolling_zscore(s, CONFIG.ZSCORE_WINDOW, CONFIG.ZSCORE_MIN_PERIODS))
level_z_long = level_z.stack().rename("IntensityLevelZ").reset_index().rename(columns={"level_1": "narrative_id"})
narr_features = narr_features.merge(level_z_long, on=["date", "narrative_id"], how="outer")

print("Narrative feature panel:", narr_features.shape)
display(narr_features.head())

# Main shock matrix used for beta estimation.
main_z_col = f"ZShock_{CONFIG.MAIN_SHOCK_WINDOW}D"
zshock_main = narr_features.pivot(index="date", columns="narrative_id", values=main_z_col).sort_index().fillna(0.0)
print("zshock_main", zshock_main.shape)



## 5. テーマ残差リターン、APWC、RRGを準備

ここではテーマ側のデータを整形します。

- `residual_return`: テーマのBarra残差リターン
- `APWC_z`: テーマ内残差相関のz-score
- `RSRatio`, `RSMomentum`, `RRG_state`: RRG座標と状態

`theme_rrg.csv` がない場合は、テーマ残差リターンからRRGを計算します。


In [ ]:

# Theme residual returns
tr = theme_returns_raw.copy()
tr.columns = [c.strip() for c in tr.columns]
tr["date"] = pd.to_datetime(tr["date"])
if "theme_id" not in tr.columns:
    for alt in ["theme", "basket", "theme_name"]:
        if alt in tr.columns:
            tr = tr.rename(columns={alt: "theme_id"})
            break
if "residual_return" not in tr.columns:
    for alt in ["return", "resid_return", "ret", "epsilon"]:
        if alt in tr.columns:
            tr = tr.rename(columns={alt: "residual_return"})
            break
tr = tr[["date", "theme_id", "residual_return"]].dropna().sort_values(["theme_id", "date"])

# APWC
ap = theme_apwc_raw.copy()
ap.columns = [c.strip() for c in ap.columns]
ap["date"] = pd.to_datetime(ap["date"])
if "theme_id" not in ap.columns:
    for alt in ["theme", "basket", "theme_name"]:
        if alt in ap.columns:
            ap = ap.rename(columns={alt: "theme_id"})
            break
if "APWC" not in ap.columns and "APWC_z" not in ap.columns:
    for alt in ["apwc", "pairwise_corr", "average_pairwise_corr"]:
        if alt in ap.columns:
            ap = ap.rename(columns={alt: "APWC"})
            break
if "APWC_z" not in ap.columns:
    ap = ap.sort_values(["theme_id", "date"])
    ap["APWC_z"] = ap.groupby("theme_id")["APWC"].transform(lambda s: rolling_zscore(s, CONFIG.APWC_Z_WINDOW, CONFIG.APWC_Z_WINDOW//2))

ap = ap[[c for c in ["date", "theme_id", "APWC", "APWC_z"] if c in ap.columns]].dropna(subset=["date", "theme_id"])

# RRG, either loaded or computed
if theme_rrg_raw is not None:
    rr = theme_rrg_raw.copy()
    rr.columns = [c.strip() for c in rr.columns]
    rr["date"] = pd.to_datetime(rr["date"])
    if "theme_id" not in rr.columns:
        for alt in ["theme", "basket", "theme_name"]:
            if alt in rr.columns:
                rr = rr.rename(columns={alt: "theme_id"})
                break
    if "RRG_state" not in rr.columns:
        rr["RRG_state"] = rr.apply(lambda x: assign_rrg_state(x.get("RSRatio", np.nan), x.get("RSMomentum", np.nan), CONFIG.RRG_HYSTERESIS_BAND), axis=1)
    rr = rr[["date", "theme_id", "RSRatio", "RSMomentum", "RRG_state"]].dropna(subset=["date", "theme_id"])
else:
    # Compute RRG from residual cumulative relative performance
    ret_piv = tr.pivot(index="date", columns="theme_id", values="residual_return").sort_index().fillna(0.0)
    log_nav = ret_piv.cumsum()
    bench = log_nav.mean(axis=1)
    rs = log_nav.sub(bench, axis=0)
    rsratio = rs.apply(lambda s: rolling_zscore(s, CONFIG.RRG_RS_WINDOW, CONFIG.RRG_RS_WINDOW//2))
    rsmom_raw = rsratio.diff(CONFIG.RRG_MOM_LAG)
    rsmom = rsmom_raw.apply(lambda s: rolling_zscore(s, CONFIG.RRG_RS_WINDOW, CONFIG.RRG_RS_WINDOW//2))
    rr = pd.concat({"RSRatio": rsratio, "RSMomentum": rsmom}, axis=1)
    rr = rr.stack(level=1).reset_index().rename(columns={"level_1": "theme_id"})
    rr["RRG_state"] = rr.apply(lambda x: assign_rrg_state(x["RSRatio"], x["RSMomentum"], CONFIG.RRG_HYSTERESIS_BAND), axis=1)

# Merge base panel
panel = tr.merge(ap, on=["date", "theme_id"], how="left").merge(rr, on=["date", "theme_id"], how="left")
panel = panel.sort_values(["theme_id", "date"])

# Add residual momentum and vol
for w in CONFIG.RMOM_WINDOWS:
    panel[f"RMOM_{w}D"] = panel.groupby("theme_id")["residual_return"].transform(lambda s: s.shift(1).rolling(w, min_periods=max(5, w//2)).sum())
panel[f"Vol_{CONFIG.VOL_WINDOW}D"] = panel.groupby("theme_id")["residual_return"].transform(lambda s: s.shift(1).rolling(CONFIG.VOL_WINDOW, min_periods=CONFIG.VOL_WINDOW//2).std())

# RRG coordinates derived features
panel["DistanceToLeading"] = np.maximum(0, -panel["RSRatio"]) + np.maximum(0, -panel["RSMomentum"])
panel["DeltaRSRatio_5D"] = panel.groupby("theme_id")["RSRatio"].diff(5)
panel["DeltaRSMomentum_5D"] = panel.groupby("theme_id")["RSMomentum"].diff(5)
panel["RRGVelocity_5D"] = np.sqrt(panel["DeltaRSRatio_5D"]**2 + panel["DeltaRSMomentum_5D"]**2)
panel["RRGAngle"] = np.arctan2(panel["RSMomentum"], panel["RSRatio"])
panel["RRGAngularVelocity_5D"] = panel.groupby("theme_id")["RRGAngle"].diff(5)

# State duration
panel["state_change"] = panel.groupby("theme_id")["RRG_state"].transform(lambda s: s.ne(s.shift()).cumsum())
panel["StateDuration"] = panel.groupby(["theme_id", "state_change"]).cumcount() + 1

print("Base theme panel", panel.shape)
display(panel.head())



## 6. テーマ別 rolling narrative beta と Narrative Signal を推定

テーマ残差リターンを、narrative shock に対して rolling ridge regression で推定します。

\[
\epsilon_{g,t} = \alpha_g + \sum_n \beta_{g,n,t} ZShock_{n,t} + u_{g,t}
\]

その後、以下を作ります。

\[
NarrativeSignal_{g,t} = \sum_n \beta_{g,n,t} ZShock_{n,t}
\]

このシグナルは、テーマにとってニュース上の追い風・逆風を表す signed signal です。


In [ ]:

# Estimate rolling narrative beta by theme.
def estimate_rolling_beta_for_theme(theme_id, ret_series, X, window, min_obs, ridge_alpha, refit_step=20):
    """
    Rolling ridge estimation with periodic refits.

    毎日betaを再推定すると重いため、デフォルトでは refit_step 営業日ごとに再推定し、
    その間は直近betaを使って NarrativeSignal を毎日計算する。
    """
    df = pd.concat([ret_series.rename("y"), X], axis=1).dropna()
    if len(df) < min_obs:
        return pd.DataFrame(), pd.DataFrame()
    Xdf = df.drop(columns=["y"])
    dates = df.index
    beta_records = []
    signal_records = []
    model = Ridge(alpha=ridge_alpha, fit_intercept=True)
    last_beta = None
    last_refit_date = None

    for i in range(len(df)):
        if i < min_obs:
            continue
        need_refit = (last_beta is None) or ((i - min_obs) % refit_step == 0)
        if need_refit:
            start = max(0, i - window)
            train = df.iloc[start:i]  # up to t-1
            if len(train) >= min_obs:
                X_train = train.drop(columns=["y"]).values
                y_train = train["y"].values
                mu = X_train.mean(axis=0)
                sd = X_train.std(axis=0)
                valid = sd > 1e-8
                if valid.sum() > 0:
                    X_train_std = (X_train[:, valid] - mu[valid]) / sd[valid]
                    try:
                        model.fit(X_train_std, y_train)
                        beta = np.zeros(Xdf.shape[1])
                        beta[valid] = model.coef_ / sd[valid]
                        last_beta = beta
                        last_refit_date = dates[i]
                        # beta records only on refit date
                        for n, b in zip(Xdf.columns, beta):
                            if abs(b) > 1e-12:
                                beta_records.append({"date": dates[i], "theme_id": theme_id, "narrative_id": n, "beta": b})
                    except Exception:
                        pass
        if last_beta is not None:
            current_x = Xdf.iloc[i].values
            signal = float(np.dot(last_beta, current_x))
            signal_records.append({
                "date": dates[i],
                "theme_id": theme_id,
                "NarrativeSignal": signal,
                "beta_refit_date": last_refit_date,
            })
    return pd.DataFrame(beta_records), pd.DataFrame(signal_records)

ret_piv = tr.pivot(index="date", columns="theme_id", values="residual_return").sort_index()
common_dates = ret_piv.index.intersection(zshock_main.index)
ret_piv = ret_piv.loc[common_dates]
zX = zshock_main.loc[common_dates].fillna(0.0)

beta_frames = []
signal_frames = []
for th in ret_piv.columns:
    bdf, sdf = estimate_rolling_beta_for_theme(th, ret_piv[th], zX, CONFIG.BETA_WINDOW, CONFIG.BETA_MIN_OBS, CONFIG.RIDGE_ALPHA, CONFIG.BETA_REFIT_STEP)
    if not bdf.empty:
        beta_frames.append(bdf)
    if not sdf.empty:
        signal_frames.append(sdf)

beta_panel = pd.concat(beta_frames, ignore_index=True) if beta_frames else pd.DataFrame(columns=["date", "theme_id", "narrative_id", "beta"])
narr_signal = pd.concat(signal_frames, ignore_index=True) if signal_frames else pd.DataFrame(columns=["date", "theme_id", "NarrativeSignal"])

# Add cross-sectional z-score and high/low flags later after merge.
print("beta_panel", beta_panel.shape)
print("narr_signal", narr_signal.shape)
display(narr_signal.head())
display(beta_panel.head())



## 7. 状態遷移分析用パネルを作成

RRG状態、APWC、Residual Momentum、Narrative Signal、将来状態、将来残差リターンを1つのパネルにまとめます。

ここで作る主な目的変数は以下です。

- `LeadEntry_hD`: 非LeadingからLeadingに入るか
- `ImpLead_hD`: ImprovingからLeadingに入るか
- `LeadExit_hD`: LeadingからWeakening/Laggingに落ちるか
- `FalseLead_H`: Leadingに入ったが持続しないか
- `FutureState_hD`: h日後のRRG状態


In [ ]:

analysis = panel.merge(narr_signal, on=["date", "theme_id"], how="left")
analysis = analysis.sort_values(["theme_id", "date"])

# Cross-sectional standardizations and flags
analysis["NarrativeSignal_xz"] = xsec_zscore(analysis, "NarrativeSignal")
analysis["APWC_xz"] = xsec_zscore(analysis, "APWC_z")
analysis["RMOM60_xz"] = xsec_zscore(analysis, "RMOM_60D")
analysis["Vol60_xz"] = xsec_zscore(analysis, f"Vol_{CONFIG.VOL_WINDOW}D")
analysis["NarrativeHigh"] = analysis.groupby("date")["NarrativeSignal"].transform(lambda s: s >= s.median()).astype(float)
analysis["APWCHigh"] = analysis.groupby("date")["APWC_z"].transform(lambda s: s >= s.median()).astype(float)

# Forward residual returns
for h in CONFIG.HORIZONS:
    analysis[f"FwdRet_{h}D"] = analysis.groupby("theme_id")["residual_return"].transform(lambda s: s.shift(-1).rolling(h, min_periods=h).sum().shift(-(h-1)))
    analysis[f"FutureState_{h}D"] = analysis.groupby("theme_id")["RRG_state"].shift(-h)
    analysis[f"Transition_{h}D"] = analysis["RRG_state"].astype(str) + " -> " + analysis[f"FutureState_{h}D"].astype(str)
    analysis[f"LeadEntry_{h}D"] = ((analysis["RRG_state"] != "Leading") & (analysis[f"FutureState_{h}D"] == "Leading")).astype(float)
    analysis[f"ImpLead_{h}D"] = ((analysis["RRG_state"] == "Improving") & (analysis[f"FutureState_{h}D"] == "Leading")).astype(float)
    analysis[f"LeadExit_{h}D"] = ((analysis["RRG_state"] == "Leading") & (analysis[f"FutureState_{h}D"].isin(["Weakening", "Lagging"]))).astype(float)

# Leading persistence and false leading over main horizon
H = CONFIG.MAIN_HORIZON
K = max(3, H // 2)
lead_indicator = (analysis["RRG_state"] == "Leading").astype(float)
analysis[f"FutureLeadingDays_{H}D"] = analysis.groupby("theme_id")["RRG_state"].transform(
    lambda s: (s.shift(-1).eq("Leading")).rolling(H, min_periods=H).sum().shift(-(H-1))
)
analysis[f"LeadPersist_{H}D"] = ((analysis["RRG_state"] == "Leading") & (analysis[f"FutureLeadingDays_{H}D"] >= K)).astype(float)
analysis[f"FalseLead_{H}D"] = ((analysis["RRG_state"] == "Leading") & (analysis[f"FutureLeadingDays_{H}D"] < K)).astype(float)

# Interactions
analysis["Narrative_x_APWC"] = analysis["NarrativeSignal_xz"] * analysis["APWC_xz"]
analysis["Improving"] = (analysis["RRG_state"] == "Improving").astype(float)
analysis["Leading"] = (analysis["RRG_state"] == "Leading").astype(float)
analysis["Improving_x_Narrative"] = analysis["Improving"] * analysis["NarrativeSignal_xz"]
analysis["Improving_x_APWC"] = analysis["Improving"] * analysis["APWC_xz"]
analysis["Improving_x_Narrative_x_APWC"] = analysis["Improving"] * analysis["NarrativeSignal_xz"] * analysis["APWC_xz"]
analysis["Leading_x_Narrative"] = analysis["Leading"] * analysis["NarrativeSignal_xz"]
analysis["Leading_x_APWC"] = analysis["Leading"] * analysis["APWC_xz"]
analysis["Leading_x_Narrative_x_APWC"] = analysis["Leading"] * analysis["NarrativeSignal_xz"] * analysis["APWC_xz"]

# Current state dummies for generic models
for st in ["Leading", "Weakening", "Lagging", "Improving"]:
    analysis[f"State_{st}"] = (analysis["RRG_state"] == st).astype(float)

print("analysis panel", analysis.shape)
display(analysis.head())



## 8. 条件付き遷移行列

まずはモデルを使わず、状態遷移確率を確認します。

特に見るべきは以下です。

\[
Pr(Improving \to Leading \mid NarrativeHigh, APWCHigh)
\]

この確率が、Narrative Low / APWC Low より高ければ、Narrative Signal と APWC がRRG状態遷移の質を分けている可能性があります。


In [ ]:

def transition_table(df, h=20, group_cols=None):
    if group_cols is None:
        group_cols = []
    d = df.dropna(subset=["RRG_state", f"FutureState_{h}D"]).copy()
    out = []
    for keys, sub in d.groupby(group_cols, dropna=False) if group_cols else [((), d)]:
        ct = pd.crosstab(sub["RRG_state"], sub[f"FutureState_{h}D"], normalize="index")
        ct_count = pd.crosstab(sub["RRG_state"], sub[f"FutureState_{h}D"])
        for cur in ct.index:
            for fut in ct.columns:
                rec = {"current_state": cur, "future_state": fut, "prob": ct.loc[cur, fut], "count": ct_count.loc[cur, fut], "horizon": h}
                if group_cols:
                    if not isinstance(keys, tuple):
                        keys = (keys,)
                    rec.update(dict(zip(group_cols, keys)))
                out.append(rec)
    return pd.DataFrame(out)

h = CONFIG.MAIN_HORIZON
overall_transitions = transition_table(analysis, h=h)
by_narrative = transition_table(analysis, h=h, group_cols=["NarrativeHigh"])
by_apwc = transition_table(analysis, h=h, group_cols=["APWCHigh"])
by_both = transition_table(analysis, h=h, group_cols=["NarrativeHigh", "APWCHigh"])

print("Overall transition probabilities")
display(overall_transitions.pivot_table(index="current_state", columns="future_state", values="prob"))

print("Improving -> Leading by NarrativeHigh / APWCHigh")
imp_lead_cond = by_both[(by_both["current_state"] == "Improving") & (by_both["future_state"] == "Leading")].copy()
display(imp_lead_cond.sort_values(["NarrativeHigh", "APWCHigh"]))



## 9. Improving → Leading ロジット

ここでは現在 `Improving` のテーマだけを対象に、将来 `Leading` に入る確率を推定します。

中核モデルは以下です。

\[
Pr(S_{g,t+h}=Leading \mid S_{g,t}=Improving)
= logit^{-1}(\beta_1 NarrativeSignal + \beta_2 APWC + \beta_3 RMOM + \beta_4 NarrativeSignal \times APWC + \cdots)
\]

重要なのは `NarrativeSignal_xz` と `Narrative_x_APWC` の係数です。


In [ ]:

h = CONFIG.MAIN_HORIZON
imp = analysis[analysis["RRG_state"] == "Improving"].copy()
y_col = f"ImpLead_{h}D"
predictors_imp = [
    "NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Vol60_xz",
    "Narrative_x_APWC", "RSRatio", "RSMomentum", "DistanceToLeading", "StateDuration"
]
model_imp, d_imp = add_constant_and_fit_glm_binomial(imp, y_col, predictors_imp, add_theme_fe=CONFIG.INCLUDE_THEME_FE, add_month_fe=CONFIG.INCLUDE_MONTH_FE)
print(f"Improving sample size used: {len(d_imp)}, positives: {d_imp[y_col].sum() if len(d_imp) else 0}")
print_key_coefs(model_imp, keys=["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Narrative_x_APWC", "RSRatio", "RSMomentum", "DistanceToLeading", "StateDuration"])



## 10. 非Leading → Leading ロジット

現在 `Leading` ではないすべてのテーマを対象に、将来 `Leading` に入るかを推定します。

Improving限定モデルより一般的ですが、Lagging や Weakening からの復帰も含むため、解釈は少し広くなります。


In [ ]:

nonlead = analysis[analysis["RRG_state"] != "Leading"].copy()
y_col = f"LeadEntry_{h}D"
predictors_nonlead = [
    "NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Vol60_xz", "Narrative_x_APWC",
    "RSRatio", "RSMomentum", "DistanceToLeading", "StateDuration",
    "State_Improving", "State_Weakening"  # Lagging基準に近い
]
model_nonlead, d_nonlead = add_constant_and_fit_glm_binomial(nonlead, y_col, predictors_nonlead, add_theme_fe=CONFIG.INCLUDE_THEME_FE, add_month_fe=CONFIG.INCLUDE_MONTH_FE)
print(f"Non-leading sample size used: {len(d_nonlead)}, positives: {d_nonlead[y_col].sum() if len(d_nonlead) else 0}")
print_key_coefs(model_nonlead, keys=["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Narrative_x_APWC", "State_Improving", "State_Weakening", "DistanceToLeading"])



## 11. Leading persistence / False Leading / Leading exit

ここでは、Narrative Signal が以下を説明できるかを見ます。

1. `Leading` が持続するか
2. `False Leading`、つまりすぐにLeadingから外れるケースを減らせるか
3. `Leading` から `Weakening` / `Lagging` に退出する確率を説明できるか


In [ ]:

# Leading persistence and false leading models.
lead_current = analysis[analysis["RRG_state"] == "Leading"].copy()
common_pred_lead = ["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Vol60_xz", "Narrative_x_APWC", "RSRatio", "RSMomentum", "StateDuration"]

for target in [f"LeadPersist_{h}D", f"FalseLead_{h}D", f"LeadExit_{h}D"]:
    print("\nTarget:", target)
    model, d = add_constant_and_fit_glm_binomial(lead_current, target, common_pred_lead, add_theme_fe=CONFIG.INCLUDE_THEME_FE, add_month_fe=CONFIG.INCLUDE_MONTH_FE)
    print(f"sample={len(d)}, positives={d[target].sum() if len(d) else 0}")
    print_key_coefs(model, keys=["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Narrative_x_APWC", "StateDuration"])



## 12. Multinomial Logit：次期RRG状態の多項分類

ここでは、将来のRRG状態そのものを4状態分類として予測します。

Narrative Signal が `Leading` 確率を上げ、`Lagging` 確率を下げるかを確認します。

このセルは scikit-learn の multinomial logistic regression を使います。係数は標準化特徴量ベースです。


In [ ]:

h = CONFIG.MAIN_HORIZON
multi_cols = ["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Vol60_xz", "RSRatio", "RSMomentum", "DistanceToLeading", "StateDuration"]
mdf = analysis.dropna(subset=multi_cols + [f"FutureState_{h}D", "RRG_state"]).copy()
# Include current state dummies explicitly
state_dummies = pd.get_dummies(mdf["RRG_state"], prefix="cur", dtype=float)
X_multi = pd.concat([mdf[multi_cols].reset_index(drop=True), state_dummies.reset_index(drop=True)], axis=1)
y_multi = mdf[f"FutureState_{h}D"].astype(str).values

if len(np.unique(y_multi)) >= 2 and len(mdf) > 100:
    clf_multi = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, solver="lbfgs"))
    ])
    clf_multi.fit(X_multi, y_multi)
    classes = clf_multi.named_steps["clf"].classes_
    coef = pd.DataFrame(clf_multi.named_steps["clf"].coef_, index=classes, columns=X_multi.columns)
    print("Multinomial classes:", classes)
    display(coef[[c for c in ["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "RSRatio", "RSMomentum"] if c in coef.columns]])
else:
    print("Not enough data/classes for multinomial model.")



## 13. Dynamic Markov Transition Model：Baseline vs Narrative-augmented

ここでは、現在のRRG状態だけで遷移確率を予測するベースラインモデルと、Narrative Signal / APWC / RMOM を追加したモデルを比較します。

比較するもの：

- in-sample log loss
- accuracy
- 将来的には out-of-sample log loss

Narrative-augmented model の log loss が低ければ、Narrative Signal は状態遷移の説明力を追加している可能性があります。


In [ ]:

# Baseline Markov probabilities by current state
markov_df = analysis.dropna(subset=["RRG_state", f"FutureState_{h}D"]).copy()
states = sorted(markov_df["RRG_state"].dropna().unique())
future_states = sorted(markov_df[f"FutureState_{h}D"].dropna().unique())

# Baseline prediction: empirical transition probabilities by current state
trans_prob = pd.crosstab(markov_df["RRG_state"], markov_df[f"FutureState_{h}D"], normalize="index")
for s in future_states:
    if s not in trans_prob.columns:
        trans_prob[s] = 0.0
trans_prob = trans_prob[future_states]

baseline_probs = np.vstack([trans_prob.loc[cur].values if cur in trans_prob.index else np.ones(len(future_states))/len(future_states) for cur in markov_df["RRG_state"]])
y_true = markov_df[f"FutureState_{h}D"].astype(str).values
label_to_idx = {s:i for i,s in enumerate(future_states)}
try:
    baseline_ll = log_loss(y_true, baseline_probs, labels=future_states)
except Exception as e:
    baseline_ll = np.nan

# Augmented multinomial model in-sample
aug_cols = ["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Vol60_xz", "RSRatio", "RSMomentum", "DistanceToLeading", "StateDuration"]
aug_df = markov_df.dropna(subset=aug_cols).copy()
state_d = pd.get_dummies(aug_df["RRG_state"], prefix="cur", dtype=float)
X_aug = pd.concat([aug_df[aug_cols].reset_index(drop=True), state_d.reset_index(drop=True)], axis=1)
y_aug = aug_df[f"FutureState_{h}D"].astype(str).values

if len(np.unique(y_aug)) >= 2 and len(aug_df) > 100:
    aug_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, solver="lbfgs"))
    ])
    aug_model.fit(X_aug, y_aug)
    pred_probs = aug_model.predict_proba(X_aug)
    aug_ll = log_loss(y_aug, pred_probs, labels=aug_model.named_steps["clf"].classes_)
    aug_acc = (aug_model.predict(X_aug) == y_aug).mean()
else:
    aug_ll, aug_acc = np.nan, np.nan

markov_compare = pd.DataFrame({
    "model": ["Baseline Markov", "Narrative-Augmented Markov"],
    "log_loss": [baseline_ll, aug_ll],
    "accuracy": [np.nan, aug_acc],
})
display(markov_compare)



## 14. Discrete-time hazard model：Leading入りまでの時間

現在Leadingではないテーマを対象に、一定ホライズン内にLeading入りする確率をハザードとして扱います。

ここでは簡略に、`LeadEntry_hD` をイベントとして使います。

重要なのは、Narrative Signal が高いほど Leading 入りのハザードが高まるかです。


In [ ]:

hazard_df = analysis[analysis["RRG_state"] != "Leading"].copy()
hazard_df["DurationNonLeading"] = hazard_df.groupby("theme_id")["RRG_state"].transform(lambda s: s.eq("Leading").cumsum())
# Better non-leading spell duration
hazard_df["nl_change"] = hazard_df.groupby("theme_id")["RRG_state"].transform(lambda s: (s.eq("Leading") | s.eq("Leading").shift(fill_value=False)).cumsum())
hazard_df["DurationNonLeading"] = hazard_df.groupby(["theme_id", "nl_change"]).cumcount() + 1

hazard_predictors = ["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Vol60_xz", "Narrative_x_APWC", "DistanceToLeading", "DurationNonLeading", "State_Improving", "State_Weakening"]
model_hazard, d_hazard = add_constant_and_fit_glm_binomial(hazard_df, f"LeadEntry_{h}D", hazard_predictors, add_theme_fe=CONFIG.INCLUDE_THEME_FE, add_month_fe=CONFIG.INCLUDE_MONTH_FE)
print(f"Hazard sample={len(d_hazard)}, events={d_hazard[f'LeadEntry_{h}D'].sum() if len(d_hazard) else 0}")
print_key_coefs(model_hazard, keys=["NarrativeSignal_xz", "APWC_xz", "Narrative_x_APWC", "DistanceToLeading", "DurationNonLeading", "State_Improving"])



## 15. Competing Risks：Improvingの成否

現在 `Improving` のテーマについて、将来どの状態に移るかを多項分類します。

特に見たいのは、Narrative Signal が

- `Improving → Leading` を増やすか
- `Improving → Lagging` を減らすか

です。


In [ ]:

cr = analysis[analysis["RRG_state"] == "Improving"].dropna(subset=[f"FutureState_{h}D"] + predictors_imp).copy()
if len(cr) > 100 and cr[f"FutureState_{h}D"].nunique() >= 2:
    X_cr = cr[predictors_imp].copy()
    y_cr = cr[f"FutureState_{h}D"].astype(str).values
    cr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, solver="lbfgs"))
    ])
    cr_model.fit(X_cr, y_cr)
    cr_classes = cr_model.named_steps["clf"].classes_
    cr_coef = pd.DataFrame(cr_model.named_steps["clf"].coef_, index=cr_classes, columns=X_cr.columns)
    display(cr_coef[[c for c in ["NarrativeSignal_xz", "APWC_xz", "Narrative_x_APWC", "RSRatio", "RSMomentum"] if c in cr_coef.columns]])

    # Probability comparison by NarrativeHigh/APWCHigh
    cr_prob = cr.copy()
    probs = cr_model.predict_proba(X_cr)
    for i, cls in enumerate(cr_classes):
        cr_prob[f"P_{cls}"] = probs[:, i]
    cols_prob = [c for c in cr_prob.columns if c.startswith("P_")]
    display(cr_prob.groupby(["NarrativeHigh", "APWCHigh"])[cols_prob].mean())
else:
    print("Not enough Improving samples/classes for competing risks model.")



## 16. Local Projection：Narrative Signal 後のRRG座標変化

ここでは、Narrative Signal が将来のRRG座標をどう動かすかを確認します。

\[
RSRatio_{g,t+h} - RSRatio_{g,t} = \alpha_g + \lambda_t + \beta_h NarrativeSignal_{g,t} + \cdots
\]

\[
RSMomentum_{g,t+h} - RSMomentum_{g,t} = \alpha_g + \lambda_t + \beta_h NarrativeSignal_{g,t} + \cdots
\]

理想的には、Narrative Signal の後にまず `RSMomentum` が上がり、その後 `RSRatio` が上がるパターンが確認されます。


In [ ]:

lp_rows = []
lp_horizons = [1, 5, 10, 20, 40, 60]
base_lp_cols = ["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Vol60_xz", "RSRatio", "RSMomentum"]
for hh in lp_horizons:
    analysis[f"DeltaRSRatio_fwd_{hh}D"] = analysis.groupby("theme_id")["RSRatio"].shift(-hh) - analysis["RSRatio"]
    analysis[f"DeltaRSMomentum_fwd_{hh}D"] = analysis.groupby("theme_id")["RSMomentum"].shift(-hh) - analysis["RSMomentum"]
    for target in [f"DeltaRSRatio_fwd_{hh}D", f"DeltaRSMomentum_fwd_{hh}D"]:
        model_lp, d_lp = fit_ols_hc1(analysis, target, base_lp_cols, add_theme_fe=CONFIG.INCLUDE_THEME_FE, add_month_fe=CONFIG.INCLUDE_MONTH_FE)
        if model_lp is not None and "NarrativeSignal_xz" in model_lp.params.index:
            lp_rows.append({
                "horizon": hh,
                "target": target,
                "coef_NarrativeSignal": model_lp.params["NarrativeSignal_xz"],
                "t_NarrativeSignal": model_lp.tvalues["NarrativeSignal_xz"],
                "n": len(d_lp),
            })

lp_results = pd.DataFrame(lp_rows)
display(lp_results)

if not lp_results.empty:
    plt.figure(figsize=(10, 5))
    for target, sub in lp_results.groupby("target"):
        plt.plot(sub["horizon"], sub["coef_NarrativeSignal"], marker="o", label=target)
    plt.axhline(0, color="black", linewidth=1)
    plt.title("Local projection: effect of NarrativeSignal on future RRG coordinate changes")
    plt.xlabel("horizon in business days")
    plt.ylabel("coefficient on NarrativeSignal_xz")
    plt.legend()
    plt.show()
plt.close()



## 17. 媒介分析：Narrative Signal → APWC → Leading 遷移

ここでは、Narrative Signal が直接RRG遷移を動かすだけでなく、まずAPWCを高め、その後Leading遷移につながるかを確認します。

手順：

1. Narrative Signal が将来 APWC を予測するか
2. 将来 APWC が Leading 遷移を予測するか
3. Narrative Signal と将来 APWC を同時に入れたとき、Narrative Signal の係数が弱まるか


In [ ]:

med_h1 = 10
med_h2 = CONFIG.MAIN_HORIZON
analysis[f"APWCZ_fwd_{med_h1}D"] = analysis.groupby("theme_id")["APWC_z"].shift(-med_h1)

# Step 1: future APWC on narrative
model_med1, d_med1 = fit_ols_hc1(
    analysis,
    f"APWCZ_fwd_{med_h1}D",
    ["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Vol60_xz"],
    add_theme_fe=CONFIG.INCLUDE_THEME_FE, add_month_fe=CONFIG.INCLUDE_MONTH_FE
)
print("Step 1: Future APWC on Narrative")
print_key_coefs(model_med1, keys=["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz"])

# Step 2: leading entry on future APWC only, in non-leading sample
med_df = analysis[analysis["RRG_state"] != "Leading"].copy()
model_med2, d_med2 = add_constant_and_fit_glm_binomial(
    med_df,
    f"LeadEntry_{med_h2}D",
    [f"APWCZ_fwd_{med_h1}D", "NarrativeSignal_xz", "RMOM60_xz", "Vol60_xz", "DistanceToLeading"],
    add_theme_fe=CONFIG.INCLUDE_THEME_FE, add_month_fe=CONFIG.INCLUDE_MONTH_FE
)
print("\nStep 2/3: Leading entry on Narrative and future APWC")
print_key_coefs(model_med2, keys=["NarrativeSignal_xz", f"APWCZ_fwd_{med_h1}D", "RMOM60_xz", "DistanceToLeading"])



## 18. 状態遷移予測と将来残差リターンの接続

状態遷移を予測できても、それが将来残差リターンにつながらなければ投資価値は限定的です。

ここでは、Leading 遷移予測確率が高いテーマほど、将来の残差リターンが高いかを検証します。


In [ ]:

# Use non-leading model to get fitted probabilities in-sample.
ret_pred_df = d_nonlead.copy() if 'd_nonlead' in globals() else pd.DataFrame()
if model_nonlead is not None and not ret_pred_df.empty:
    # Reconstruct design used in model_nonlead with same helper logic for prediction.
    # Simpler: use model fitted values aligned to d_nonlead.
    ret_pred_df = ret_pred_df.copy()
    ret_pred_df["PredLeadProb"] = np.asarray(model_nonlead.fittedvalues)
    # Attach forward return and key variables from analysis.
    cols_attach = ["date", "theme_id", f"FwdRet_{h}D", "RRG_state", "NarrativeSignal_xz", "APWC_xz", "RMOM60_xz"]
    ret_pred_df = ret_pred_df.merge(analysis[cols_attach], on=["date", "theme_id"], how="left", suffixes=("", "_y"))
    model_retprob, d_retprob = fit_ols_hc1(
        ret_pred_df,
        f"FwdRet_{h}D",
        ["PredLeadProb", "NarrativeSignal_xz", "APWC_xz", "RMOM60_xz"],
        add_theme_fe=CONFIG.INCLUDE_THEME_FE, add_month_fe=CONFIG.INCLUDE_MONTH_FE
    )
    print_key_coefs(model_retprob, keys=["PredLeadProb", "NarrativeSignal_xz", "APWC_xz", "RMOM60_xz"])

    ret_pred_df["PredProbDecile"] = ret_pred_df.groupby("date")["PredLeadProb"].transform(lambda s: safe_qcut(s.rank(method="first"), 10, labels=False))
    decile_ret = ret_pred_df.groupby("PredProbDecile")[f"FwdRet_{h}D"].agg(["count", "mean", "median", lambda x: (x>0).mean()]).rename(columns={"<lambda_0>": "hit_rate"})
    display(decile_ret)
else:
    print("Non-leading model not available for transition-return connection.")



## 19. Out-of-sample 予測：Narrative Signal は状態遷移予測を改善するか

ここでは walk-forward で、以下のモデルを比較します。

1. **Baseline**: 現在状態、RRG座標、Distanceなど
2. **No Narrative**: Baseline + APWC + RMOM
3. **Full**: Baseline + APWC + RMOM + Narrative Signal + Narrative×APWC

評価指標：

- AUC
- PR-AUC
- Log loss
- Brier score
- Top-decile hit rate
- Lift

主対象は `Improving → Leading` です。


In [ ]:

def prepare_oos_dataset(df, target_col, sample_filter=None):
    d = df.copy()
    if sample_filter is not None:
        d = d.query(sample_filter).copy()
    d = d.dropna(subset=[target_col, "date", "theme_id", "NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "RSRatio", "RSMomentum", "DistanceToLeading", "StateDuration", "Vol60_xz"])
    d = d.sort_values("date")
    return d

features_baseline = ["RSRatio", "RSMomentum", "DistanceToLeading", "StateDuration", "Vol60_xz"]
features_no_narr = features_baseline + ["APWC_xz", "RMOM60_xz"]
features_full = features_no_narr + ["NarrativeSignal_xz", "Narrative_x_APWC"]

def walk_forward_binary(d, target_col, feature_sets, min_train_days=504, step_days=126, max_folds=None):
    dates = np.array(sorted(d["date"].unique()))
    rows = []
    pred_frames = []
    if len(dates) < min_train_days + step_days:
        return pd.DataFrame(), pd.DataFrame()
    fold_count = 0
    for start_idx in range(min_train_days, len(dates) - step_days + 1, step_days):
        train_end = dates[start_idx-1]
        test_start = dates[start_idx]
        test_end = dates[min(start_idx + step_days - 1, len(dates)-1)]
        train = d[d["date"] <= train_end].copy()
        test = d[(d["date"] >= test_start) & (d["date"] <= test_end)].copy()
        if train[target_col].nunique() < 2 or test[target_col].nunique() < 2:
            continue
        y_train = train[target_col].astype(int).values
        y_test = test[target_col].astype(int).values
        for name, feats in feature_sets.items():
            X_train = train[feats].values
            X_test = test[feats].values
            clf = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=500, solver="liblinear"))
            ])
            try:
                clf.fit(X_train, y_train)
                p = clf.predict_proba(X_test)[:,1]
                auc = roc_auc_score(y_test, p) if len(np.unique(y_test)) > 1 else np.nan
                ap = average_precision_score(y_test, p) if len(np.unique(y_test)) > 1 else np.nan
                ll = log_loss(y_test, p, labels=[0,1])
                br = brier_score_loss(y_test, p)
                # Top decile hit rate by date cross-section
                tmp = test[["date", "theme_id", target_col, f"FwdRet_{CONFIG.MAIN_HORIZON}D"]].copy()
                tmp["pred_prob"] = p
                tmp["model"] = name
                tmp["fold_train_end"] = train_end
                pred_frames.append(tmp)
                top = tmp.groupby("date").apply(lambda x: x.nlargest(max(1, int(np.ceil(len(x)*0.1))), "pred_prob")).reset_index(drop=True)
                top_hit = top[target_col].mean() if len(top) else np.nan
                base_hit = tmp[target_col].mean() if len(tmp) else np.nan
                lift = top_hit / base_hit if base_hit and not np.isnan(base_hit) else np.nan
                rows.append({"model": name, "train_end": train_end, "test_start": test_start, "test_end": test_end, "auc": auc, "pr_auc": ap, "log_loss": ll, "brier": br, "top_decile_hit": top_hit, "base_hit": base_hit, "lift": lift, "n_test": len(test)})
            except Exception as e:
                rows.append({"model": name, "train_end": train_end, "test_start": test_start, "test_end": test_end, "error": str(e)})
        fold_count += 1
        if max_folds is not None and fold_count >= max_folds:
            break
    return pd.DataFrame(rows), pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()

target_oos = f"ImpLead_{h}D"
oos_data = prepare_oos_dataset(analysis, target_oos, sample_filter="RRG_state == 'Improving'")
feature_sets = {"Baseline": features_baseline, "NoNarrative": features_no_narr, "FullNarrative": features_full}
oos_metrics, oos_preds = walk_forward_binary(oos_data, target_oos, feature_sets, CONFIG.TRAIN_MIN_DAYS, CONFIG.TEST_STEP_DAYS, CONFIG.OOS_MAX_FOLDS)

if not oos_metrics.empty:
    display(oos_metrics.groupby("model")[["auc", "pr_auc", "log_loss", "brier", "top_decile_hit", "base_hit", "lift", "n_test"]].mean())
else:
    print("Not enough data for OOS walk-forward prediction.")



## 20. 経済評価：遷移予測確率を使ったテーマ選択

Out-of-sample 予測確率が得られている場合、予測確率上位の Improving テーマを保有する簡易戦略を評価します。

ここでは、将来Leading化しやすいと予測されたテーマが、実際に翌日以降の残差リターンを生むかを確認します。


In [ ]:

if 'oos_preds' in globals() and not oos_preds.empty:
    # Use FullNarrative model predictions for strategy.
    pred = oos_preds[oos_preds["model"] == "FullNarrative"].copy()
    next_ret = analysis[["date", "theme_id", "residual_return"]].copy()
    next_ret["NextRet1D"] = next_ret.groupby("theme_id")["residual_return"].shift(-1)
    pred = pred.merge(next_ret[["date", "theme_id", "NextRet1D"]], on=["date", "theme_id"], how="left")

    # Select top 20% predicted transition-probability themes each date.
    pred["prob_cutoff"] = pred.groupby("date")["pred_prob"].transform(lambda s: s.quantile(0.80))
    pred["selected"] = pred["pred_prob"] >= pred["prob_cutoff"]
    strat = pred[pred["selected"]].groupby("date")["NextRet1D"].mean().rename("TransitionProbTopStrategy").reset_index()
    stats = ann_stats(strat["TransitionProbTopStrategy"])
    display(stats.to_frame("TransitionProbTopStrategy").T)

    # Cumulative return table; plot can be enabled manually if desired.
    strat["cum_return"] = (1 + strat["TransitionProbTopStrategy"].fillna(0)).cumprod()
    display(strat.tail())
else:
    print("OOS predictions unavailable. Run OOS section with enough data to evaluate economic strategy.")



## 21. Robustness hooks：ホライズン別・シグナル別の比較

このセルでは、主要ホライズンごとに、Narrative Signal が Leading 遷移を説明する係数をざっくり比較します。

詳細なロバストネスでは、以下も比較してください。

- 20D / 60D / 120D の narrative shock
- negative / total / positive intensity
- OLS beta / ridge beta
- テーマ直接 beta / 構成銘柄 beta 集約
- 日次RRG / 週次RRG
- hysteresis band あり / なし


In [ ]:

robust_rows = []
robust_features = ["NarrativeSignal_xz", "APWC_xz", "RMOM60_xz", "Narrative_x_APWC", "RSRatio", "RSMomentum", "DistanceToLeading"]
for hh in CONFIG.HORIZONS:
    target = f"ImpLead_{hh}D"
    if target not in analysis.columns:
        continue
    d = analysis[analysis["RRG_state"] == "Improving"].dropna(subset=[target] + robust_features).copy()
    if len(d) < 100 or d[target].nunique() < 2:
        continue
    X = d[robust_features].values
    y = d[target].astype(int).values
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500, solver="liblinear"))
    ])
    try:
        clf.fit(X, y)
        coefs = clf.named_steps["clf"].coef_[0]
        p = clf.predict_proba(X)[:, 1]
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan
        for name, coef in zip(robust_features, coefs):
            if name in ["NarrativeSignal_xz", "APWC_xz", "Narrative_x_APWC"]:
                robust_rows.append({
                    "horizon": hh,
                    "feature": name,
                    "standardized_logit_coef": coef,
                    "in_sample_auc": auc,
                    "n": len(d),
                    "events": y.sum(),
                })
    except Exception as e:
        robust_rows.append({"horizon": hh, "feature": "ERROR", "error": str(e), "n": len(d), "events": y.sum()})

robust_table = pd.DataFrame(robust_rows)
display(robust_table)



## 22. Narrative beta の診断

最後に、テーマごとにどの narrative beta が大きいかを確認します。

これは結果の経済的解釈に必須です。

- 正の beta: その narrative shock はテーマに追い風
- 負の beta: その narrative shock はテーマに逆風
- 絶対値が大きい: テーマがその narrative に強く反応


In [ ]:

if not beta_panel.empty:
    latest_date = beta_panel["date"].max()
    latest_beta = beta_panel[beta_panel["date"] == latest_date].copy()
    latest_beta["abs_beta"] = latest_beta["beta"].abs()
    top_beta = latest_beta.sort_values(["theme_id", "abs_beta"], ascending=[True, False]).groupby("theme_id").head(5)
    print("Latest beta date:", latest_date)
    display(top_beta)

    # Heatmap for top narratives by aggregate abs beta.
    top_ns = latest_beta.groupby("narrative_id")["abs_beta"].mean().sort_values(ascending=False).head(15).index
    heat = latest_beta[latest_beta["narrative_id"].isin(top_ns)].pivot_table(index="theme_id", columns="narrative_id", values="beta", aggfunc="mean")
    plt.figure(figsize=(12, 6))
    plt.imshow(heat.fillna(0).values, aspect="auto")
    plt.colorbar(label="beta")
    plt.xticks(range(len(heat.columns)), heat.columns, rotation=90)
    plt.yticks(range(len(heat.index)), heat.index)
    plt.title("Latest theme narrative beta heatmap")
    plt.tight_layout()
    plt.show()
    plt.close()
else:
    print("beta_panel is empty.")



## 23. 保存

`CONFIG.SAVE_OUTPUTS=True` の場合、主要な中間結果と分析結果をCSVとして保存します。


In [ ]:

if CONFIG.SAVE_OUTPUTS:
    outdir = Path(CONFIG.OUTPUT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    analysis.to_csv(outdir / "rrg_transition_narrative_beta_analysis_panel.csv", index=False)
    beta_panel.to_csv(outdir / "theme_narrative_beta_panel.csv", index=False)
    narr_signal.to_csv(outdir / "theme_narrative_signal.csv", index=False)
    overall_transitions.to_csv(outdir / "transition_probabilities_overall.csv", index=False)
    by_both.to_csv(outdir / "transition_probabilities_by_narrative_apwc.csv", index=False)
    if 'robust_table' in globals():
        robust_table.to_csv(outdir / "robustness_imp_to_lead_logit.csv", index=False)
    if 'oos_metrics' in globals() and not oos_metrics.empty:
        oos_metrics.to_csv(outdir / "oos_transition_prediction_metrics.csv", index=False)
    if 'oos_preds' in globals() and not oos_preds.empty:
        oos_preds.to_csv(outdir / "oos_transition_predictions.csv", index=False)
    print(f"Saved outputs to {outdir}")
else:
    print("SAVE_OUTPUTS is False. No files saved.")



# 結果の解釈ガイド

このノートブックで最も重要な確認点は以下です。

## 1. 条件付き遷移行列

まず、以下が成り立つかを確認します。

\[
Pr(Improving \to Leading \mid NarrativeHigh, APWCHigh)
>
Pr(Improving \to Leading \mid NarrativeLow, APWCLow)
\]

これが成り立たない場合、複雑なモデルに進む前に、Narrative Signal または RRG 状態定義を見直すべきです。

## 2. Improving → Leading ロジット

`NarrativeSignal_xz` が正なら、Narrative Signal が Improving → Leading 遷移を説明します。

`Narrative_x_APWC` が正なら、Narrative Signal と APWC が同時に高いときに Leading 化しやすいことを意味します。

## 3. Multinomial Logit / Competing Risks

Narrative Signal が `Leading` の係数を上げ、`Lagging` の係数を下げるなら、テーマのライフサイクルを上昇方向に押し上げる情報を持つ可能性があります。

## 4. Local Projection

Narrative Signal 後に、まず `RSMomentum`、次に `RSRatio` が上昇するなら、ニュースがRRG上の回転を先行するというストーリーに整合します。

## 5. Mediation

Narrative Signal が将来APWCを上げ、APWCがLeading遷移を説明するなら、

\[
Narrative \to APWC \to RRG Leading
\]

という媒介経路が示唆されます。

## 6. Out-of-sample予測

`FullNarrative` モデルが `NoNarrative` モデルより AUC、PR-AUC、log loss、Brier score、top-decile hit rate で改善するかを見ます。

説明分析だけでなく、予測性能が改善することが重要です。

## 7. 経済評価

Transition probability 上位テーマの残差リターンが右肩上がりなら、状態遷移予測が投資価値を持つ可能性があります。

最終的に支持したい仮説は以下です。

\[
NarrativeSignal_{g,t}
\times
APWCZ_{g,t}
\Rightarrow
Pr(S_{g,t+h}=Leading \mid S_{g,t}=Improving)
\Rightarrow
R^{resid}_{g,t+1:t+h} > 0
\]
